# MedQuery Phase 9 — Full-Scale HPC Ingestion

Resumable execution notebook for Kubeflow JupyterLab (`H100 NVL MIG 3g.47gb`, 9.6 CPU cores, 38.4 GiB RAM, 30 GiB persistent `/home/jovyan`).

- **Isolated environments:** Leaves base Conda untouched; uses isolated `embedding-venv` (`--system-site-packages` inheriting container `PyTorch 2.5.1+cu124` + pinned `NumPy 1.26.4`, `SciPy 1.14.1`, `scikit-learn 1.5.2`) alongside CPU `docling-venv` and `paddle-venv`.
- **Chunking:** `section_aware` (`min=300`, `target=380`, `max=450`, `overlap=0`)
- **Embeddings:** `intfloat/multilingual-e5-small` (`rev 614241f622f53c4eeff9890bdc4f31cfecc418b3`, Safetensors, `384d`, cosine, mean pooling, L2-normalized)
- **Safety:** Zero external API calls; all outputs remain staged until local PostgreSQL governance review.

## 1. Environment Verification

In [ ]:
import os, platform, shutil, socket, sys
from pathlib import Path

def read_first(*paths):
    for p in paths:
        fp = Path(p)
        if fp.is_file():
            return fp.read_text().strip()
    return None

import torch
print("Python:", sys.version.split()[0], "| Platform:", platform.platform())
print("PyTorch:", torch.__version__, "| CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f"GPU: {props.name} | VRAM: {props.total_memory / 1024**3:.3f} GiB")

cpu_max = read_first("/sys/fs/cgroup/cpu.max")
cpu_quota = read_first("/sys/fs/cgroup/cpu/cpu.cfs_quota_us")
cpu_period = read_first("/sys/fs/cgroup/cpu/cpu.cfs_period_us")
if cpu_max and cpu_max.split()[0] != "max":
    q, p = map(int, cpu_max.split())
    print(f"CPU quota (cgroup v2): {q / p:.2f} cores (host os.cpu_count={os.cpu_count()})")
elif cpu_quota and cpu_period and int(cpu_quota) > 0:
    print(f"CPU quota (cgroup v1): {int(cpu_quota) / int(cpu_period):.2f} cores (host os.cpu_count={os.cpu_count()})")
else:
    print(f"Host os.cpu_count={os.cpu_count()}")

mem_max = read_first("/sys/fs/cgroup/memory.max", "/sys/fs/cgroup/memory/memory.limit_in_bytes")
if mem_max and mem_max.isdigit():
    print(f"RAM limit (cgroup): {int(mem_max) / 1024**3:.2f} GiB")

jovyan_usage = shutil.disk_usage("/home/jovyan" if Path("/home/jovyan").exists() else ".")
print(f"Persistent storage free: {jovyan_usage.free / 1024**3:.2f} GiB / {jovyan_usage.total / 1024**3:.2f} GiB")

for tool in ("tesseract", "pdftotext", "/usr/bin/time", "conda", "mamba"):
    print(f"Tool {tool:15s}:", shutil.which(tool) or ("present" if Path(tool).is_file() else "missing"))

for host in ("huggingface.co", "pypi.org"):
    try:
        socket.getaddrinfo(host, 443)
        print(f"DNS OK: {host}")
    except OSError as exc:
        print(f"DNS FAIL: {host} -> {exc}")

## 2. Project Setup (Code, Isolated `embedding-venv`, Tesseract, Pinned Models)

Avoids modifying the base Conda environment: creates `embedding-venv` (`--system-site-packages` so `PyTorch 2.5.1+cu124` CUDA is preserved) with `requirements-hpc.txt` and compatible `numpy==1.26.4`, `scipy==1.14.1`, `scikit-learn==1.5.2`.

In [ ]:
import os, subprocess, sys, zipfile
from pathlib import Path

WORKSPACE = Path(os.environ.get("MEDQUERY_WORKSPACE", "/home/jovyan/medquery_workspace"))
UPLOAD_DIR = Path(os.environ.get("MEDQUERY_UPLOAD_DIR", "/home/jovyan"))
REPO_DIR = WORKSPACE / "repo"
INPUT_DIR = WORKSPACE / "input"
SETUP_DIR = WORKSPACE / "setup"
HF_CACHE = WORKSPACE / "hf_cache"
for d in (WORKSPACE, REPO_DIR, INPUT_DIR, SETUP_DIR, HF_CACHE):
    d.mkdir(parents=True, exist_ok=True)

os.environ["HF_HOME"] = str(HF_CACHE)
os.environ["HF_HUB_DISABLE_TELEMETRY"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

def find_upload(name):
    for cand in (UPLOAD_DIR / name, WORKSPACE / name, Path.cwd() / name):
        if cand.is_file():
            return cand
    raise FileNotFoundError(f"Upload {name} to {UPLOAD_DIR} (or {WORKSPACE}) before running this cell.")

code_zip = find_upload("medquery_hpc_code.zip")
with zipfile.ZipFile(code_zip, "r") as zf:
    zf.extractall(REPO_DIR)
print(f"Extracted {code_zip} -> {REPO_DIR}")

# Create isolated embedding-venv inheriting container PyTorch 2.5.1+cu124 without modifying base Conda.
embed_venv = SETUP_DIR / "embedding-venv"
embed_py = embed_venv / "bin/python"
if not embed_py.is_file():
    subprocess.run([sys.executable, "-m", "venv", "--system-site-packages", str(embed_venv)], check=True)
subprocess.run([str(embed_py), "-m", "pip", "install", "-q", "--upgrade", "pip"], check=True)
subprocess.run([
    str(embed_py), "-m", "pip", "install", "-q",
    "-r", str(REPO_DIR / "requirements-hpc.txt"),
    "numpy==1.26.4", "scipy==1.14.1", "scikit-learn==1.5.2",
], check=True)
subprocess.run([
    str(embed_py), "-c",
    "import torch, numpy, scipy, sklearn; "
    "assert 'cu' in torch.__version__ or torch.cuda.is_available(), f'Unexpected PyTorch: {torch.__version__}'; "
    "print('embedding-venv ready | PyTorch:', torch.__version__, '| NumPy:', numpy.__version__, '| SciPy:', scipy.__version__, '| sklearn:', sklearn.__version__)",
], check=True)

In [ ]:
import hashlib, os, shutil, subprocess, sys
from pathlib import Path

WORKSPACE = Path(os.environ.get("MEDQUERY_WORKSPACE", "/home/jovyan/medquery_workspace"))
REPO_DIR = WORKSPACE / "repo"
SETUP_DIR = WORKSPACE / "setup"
HF_CACHE = WORKSPACE / "hf_cache"
EMBED_PY = str(SETUP_DIR / "embedding-venv/bin/python") if (SETUP_DIR / "embedding-venv/bin/python").is_file() else sys.executable
os.environ["HF_HOME"] = str(HF_CACHE)

def has_tess_langs():
    tess = shutil.which("tesseract") or os.environ.get("PHASE3_TESSERACT")
    if not tess:
        return False, set()
    args = [tess]
    if td := (os.environ.get("PHASE3_TESSDATA") or os.environ.get("TESSDATA_PREFIX")):
        args += ["--tessdata-dir", td]
    proc = subprocess.run([*args, "--list-langs"], capture_output=True, text=True)
    langs = {line.strip() for line in proc.stdout.splitlines()[1:] if line.strip()}
    return {"ara", "eng", "fra"}.issubset(langs), langs

ok_tess, langs = has_tess_langs()
if not ok_tess or not shutil.which("pdftotext"):
    pkg_mgr = shutil.which("mamba") or shutil.which("conda")
    if pkg_mgr:
        print("Installing tesseract and poppler via", pkg_mgr)
        subprocess.run([pkg_mgr, "install", "-y", "-c", "conda-forge", "tesseract", "poppler"], check=True)
    elif os.geteuid() == 0 and shutil.which("apt-get"):
        subprocess.run(["apt-get", "update", "-qq"], check=True)
        subprocess.run(["apt-get", "install", "-y", "tesseract-ocr", "tesseract-ocr-ara", "tesseract-ocr-fra", "tesseract-ocr-eng", "poppler-utils", "time"], check=True)

tess_bin = shutil.which("tesseract")
if tess_bin:
    os.environ["PHASE3_TESSERACT"] = tess_bin
    tessdata_dir = SETUP_DIR / "tessdata"
    tessdata_dir.mkdir(parents=True, exist_ok=True)
    sys_dirs = [Path("/usr/share/tesseract-ocr/5/tessdata"), Path("/usr/share/tessdata"), Path(tess_bin).resolve().parents[1] / "share/tessdata"]
    for lang in ("eng", "fra", "ara"):
        dest = tessdata_dir / f"{lang}.traineddata"
        if not dest.is_file():
            src = next((d / f"{lang}.traineddata" for d in sys_dirs if (d / f"{lang}.traineddata").is_file()), None)
            if src:
                shutil.copy2(src, dest)
            else:
                url = f"https://github.com/tesseract-ocr/tessdata_fast/raw/main/{lang}.traineddata"
                subprocess.run(["curl", "--fail", "--location", "--silent", "--show-error", url, "-o", str(dest)], check=True)
    os.environ["PHASE3_TESSDATA"] = str(tessdata_dir)
    ok_tess, langs = has_tess_langs()

assert ok_tess, f"Missing required Tesseract languages (need ara, eng, fra); got {sorted(langs)}"
assert shutil.which("pdftotext"), "Missing poppler pdftotext"
print("Tesseract langs verified:", sorted(langs & {"ara", "eng", "fra"}))

# Isolated Docling venv (CPU PyTorch inside venv; base PyTorch 2.5.1+cu124 stays untouched)
docling_venv = SETUP_DIR / "docling-venv"
docling_py = docling_venv / "bin/python"
if not docling_py.is_file():
    subprocess.run([sys.executable, "-m", "venv", str(docling_venv)], check=True)
    subprocess.run([str(docling_py), "-m", "pip", "install", "-q", "--upgrade", "pip"], check=True)
    subprocess.run([str(docling_py), "-m", "pip", "install", "-q", "--index-url", "https://download.pytorch.org/whl/cpu", "torch==2.8.0+cpu", "torchvision==0.23.0+cpu"], check=True)
    subprocess.run([str(docling_py), "-m", "pip", "install", "-q", "-r", str(REPO_DIR / "experiments/extraction/requirements-docling.txt")], check=True)
subprocess.run(["bash", str(REPO_DIR / "experiments/extraction/download_docling_models.sh")], check=True)

# Isolated PaddleOCR venv
paddle_venv = SETUP_DIR / "paddle-venv"
paddle_py = paddle_venv / "bin/python"
if not paddle_py.is_file():
    subprocess.run([sys.executable, "-m", "venv", str(paddle_venv)], check=True)
    subprocess.run([str(paddle_py), "-m", "pip", "install", "-q", "--upgrade", "pip"], check=True)
    subprocess.run([str(paddle_py), "-m", "pip", "install", "-q", "-r", str(REPO_DIR / "experiments/extraction/requirements-paddle.txt")], check=True)
subprocess.run(["bash", str(REPO_DIR / "experiments/extraction/download_paddle_models.sh")], check=True)

# Download pinned E5-small Safetensors weights + verify exact Phase 5 tokenizer via embedding-venv
dl_script = (
    "import hashlib, shutil; from pathlib import Path; from huggingface_hub import snapshot_download; "
    "E5_MODEL = 'intfloat/multilingual-e5-small'; "
    "E5_REV = '614241f622f53c4eeff9890bdc4f31cfecc418b3'; "
    "snap_dir = Path(snapshot_download(repo_id=E5_MODEL, revision=E5_REV, ignore_patterns=['*.bin', '*.h5', '*.ot'])); "
    "assert (snap_dir / 'model.safetensors').is_file(), f'Missing model.safetensors in {snap_dir}'; "
    f"tok_dest = Path({str(REPO_DIR / 'data/ingestion/phase5/tokenizer/tokenizer.json')!r}); "
    "tok_dest.parent.mkdir(parents=True, exist_ok=True); "
    "shutil.copy2(snap_dir / 'tokenizer.json', tok_dest); "
    "tok_sha = hashlib.sha256(tok_dest.read_bytes()).hexdigest(); "
    "assert tok_sha == '0b44a9d7b51c3c62626640cda0e2c2f70fdacdc25bbbd68038369d14ebdf4c39', f'Tokenizer mismatch: {tok_sha}'; "
    "print('Pinned Safetensors weights & tokenizer verified in', snap_dir)"
)
subprocess.run([EMBED_PY, "-c", dl_script], env=os.environ, check=True)

## 3. Dataset Preparation & Checksum Verification

In [ ]:
import json, os, subprocess, sys
from pathlib import Path
import zipfile

WORKSPACE = Path(os.environ.get("MEDQUERY_WORKSPACE", "/home/jovyan/medquery_workspace"))
UPLOAD_DIR = Path(os.environ.get("MEDQUERY_UPLOAD_DIR", "/home/jovyan"))
REPO_DIR = WORKSPACE / "repo"
INPUT_DIR = WORKSPACE / "input"
SETUP_DIR = WORKSPACE / "setup"
EMBED_PY = str(SETUP_DIR / "embedding-venv/bin/python") if (SETUP_DIR / "embedding-venv/bin/python").is_file() else sys.executable

input_zip = next((p for p in (UPLOAD_DIR / "medquery_hpc_input.zip", WORKSPACE / "medquery_hpc_input.zip", Path.cwd() / "medquery_hpc_input.zip") if p.is_file()), None)
if input_zip is None and not (INPUT_DIR / "manifests/checksums.sha256").is_file():
    raise FileNotFoundError(f"Upload medquery_hpc_input.zip to {UPLOAD_DIR}")

if not (INPUT_DIR / "manifests/checksums.sha256").is_file():
    with zipfile.ZipFile(input_zip, "r") as zf:
        zf.extractall(INPUT_DIR)
    print(f"Extracted {input_zip} -> {INPUT_DIR}")

subprocess.run(["sha256sum", "-c", "--quiet", "manifests/checksums.sha256"], cwd=INPUT_DIR, check=True)
print("All raw PDF and manifest SHA-256 checksums verified.")

prep_script = f'''
import json
from pathlib import Path
from ingestion.extraction.selection import rebase_selection_manifest, verify_manifest_assets
REPO_DIR = Path({str(REPO_DIR)!r})
INPUT_DIR = Path({str(INPUT_DIR)!r})
rebased_manifest = INPUT_DIR / "manifests/hpc_runtime_manifest.json"
collection_roots = {{
    "clinical_protocols_ministry_directives": str((INPUT_DIR / "raw/clinical_protocols_ministry_directives").resolve()),
    "treatment_protocols": str((INPUT_DIR / "raw/treatment_protocols").resolve()),
    "therapeutic_guidelines": str((INPUT_DIR / "raw/therapeutic_guidelines").resolve()),
}}
rebased = rebase_selection_manifest(
    INPUT_DIR / "manifests/extraction_selection_manifest.json",
    rebased_manifest,
    collection_roots,
    root=REPO_DIR,
    inventories={{}},
)
ver = verify_manifest_assets(rebased_manifest, root=REPO_DIR, check_pdf_pages=True)
assert ver["ok"], f"Asset verification failed: {{ver['issues']}}"
print(json.dumps({{
    "eligible_assets": ver["eligible_assets"],
    "unique_sha256": ver["unique_sha256"],
    "verified_pages": ver["verified_pages"],
    "verified_mib": round(ver["verified_bytes"] / 1024**2, 2),
    "excluded_or_held_assets": rebased["summary"]["excluded_or_held_assets"],
}}, indent=2))
'''
subprocess.run([EMBED_PY, "-c", prep_script], cwd=REPO_DIR, check=True)

## 4. Small HPC Smoke Test (Extraction, Chunking, CPU/CUDA Compatibility)

In [ ]:
import json, os, shutil, subprocess, sys
from pathlib import Path

WORKSPACE = Path(os.environ.get("MEDQUERY_WORKSPACE", "/home/jovyan/medquery_workspace"))
REPO_DIR = WORKSPACE / "repo"
INPUT_DIR = WORKSPACE / "input"
SETUP_DIR = WORKSPACE / "setup"
EMBED_PY = str(SETUP_DIR / "embedding-venv/bin/python") if (SETUP_DIR / "embedding-venv/bin/python").is_file() else sys.executable

def configure_runtime_env():
    env = os.environ.copy()
    env["HF_HOME"] = str(WORKSPACE / "hf_cache")
    env["HF_HUB_DISABLE_TELEMETRY"] = "1"
    env["TOKENIZERS_PARALLELISM"] = "false"
    if (SETUP_DIR / "tessdata").is_dir():
        env["PHASE3_TESSDATA"] = str(SETUP_DIR / "tessdata")
    if tess := shutil.which("tesseract"):
        env.setdefault("PHASE3_TESSERACT", tess)
    if (SETUP_DIR / "docling-venv/bin/python").is_file():
        env["PHASE3_DOCLING_PYTHON"] = str(SETUP_DIR / "docling-venv/bin/python")
    if (SETUP_DIR / "paddle-venv/bin/python").is_file():
        env["PHASE3_PADDLE_PYTHON"] = str(SETUP_DIR / "paddle-venv/bin/python")
    env["PHASE3_DOCLING_MODELS"] = str(REPO_DIR / "data/ingestion/phase3/setup/docling_models")
    env["PHASE3_PADDLE_MODELS"] = str(REPO_DIR / "data/ingestion/phase3/setup/paddle_models")
    return env

ENV = configure_runtime_env()
SMOKE_DIR = REPO_DIR / "data/ingestion/phase9/smoke_run"
MANIFEST = INPUT_DIR / "manifests/hpc_runtime_manifest.json"
PROFILE = REPO_DIR / "ingestion/extraction/profile_hpc.json"

# 4 representative pages: FR OCR/vector, FR scan+table, EN bundle, AR OCR/columns
subprocess.run([
    EMBED_PY, "-m", "ingestion", "extract",
    "--selection-manifest", str(MANIFEST),
    "--document", "MQ-CP-NEWBORN-STANDARDS-2021",
    "--document", "MQ-TG-DZ-TB",
    "--document", "TG-0010-FULL-PDF",
    "--document", "MQ-TG-WHO-EMRO-MHGAP-AR",
    "--pages", "MQ-CP-NEWBORN-STANDARDS-2021:24",
    "--pages", "MQ-TG-DZ-TB:1",
    "--pages", "TG-0010-FULL-PDF:13",
    "--pages", "MQ-TG-WHO-EMRO-MHGAP-AR:10",
    "--profile", str(PROFILE),
    "--workers", "2",
    "--resume",
    "--output", str(SMOKE_DIR / "extract"),
], cwd=REPO_DIR, env=ENV, check=True)

subprocess.run([
    EMBED_PY, "-m", "ingestion", "chunk",
    "--input", str(SMOKE_DIR / "extract"),
    "--output", str(SMOKE_DIR / "chunk"),
    "--strategy", "section_aware",
    "--min-tokens", "300", "--target-tokens", "380", "--max-tokens", "450",
], cwd=REPO_DIR, env=ENV, check=True)

subprocess.run([
    EMBED_PY, "-m", "ingestion", "embed",
    "--input", str(SMOKE_DIR / "chunk"),
    "--output", str(SMOKE_DIR / "embed"),
    "--model", "intfloat/multilingual-e5-small",
    "--device", "cuda",
    "--gpu-batch-size", "16",
    "--local-files-only",
], cwd=REPO_DIR, env=ENV, check=True)

compat_script = f'''
import json
from ingestion.chunking import read_jsonl
from ingestion.embeddings import verify_cpu_gpu_compatibility
chunks = read_jsonl({str(SMOKE_DIR / "chunk/chunks.jsonl")!r})
compat = verify_cpu_gpu_compatibility([c["retrieval_text"] for c in chunks[:8]], local_files_only=True)
print("CPU/GPU numerical compatibility:", json.dumps(compat, indent=2))
'''
subprocess.run([EMBED_PY, "-c", compat_script], cwd=REPO_DIR, env=ENV, check=True)
print("Smoke embedding summary:", json.dumps(json.loads((SMOKE_DIR / "embed/embedding_summary.json").read_text()), indent=2))

## 5. Full Extraction and Normalization (Resumable)

In [ ]:
import json, os, shutil, subprocess, sys, time
from pathlib import Path

WORKSPACE = Path(os.environ.get("MEDQUERY_WORKSPACE", "/home/jovyan/medquery_workspace"))
REPO_DIR = WORKSPACE / "repo"
INPUT_DIR = WORKSPACE / "input"
SETUP_DIR = WORKSPACE / "setup"
EMBED_PY = str(SETUP_DIR / "embedding-venv/bin/python") if (SETUP_DIR / "embedding-venv/bin/python").is_file() else sys.executable
FULL_DIR = REPO_DIR / "data/ingestion/phase9/full_run"
EXTRACT_DIR = FULL_DIR / "extract"
MANIFEST = INPUT_DIR / "manifests/hpc_runtime_manifest.json"
PROFILE = REPO_DIR / "ingestion/extraction/profile_hpc.json"

ENV = os.environ.copy()
ENV["HF_HOME"] = str(WORKSPACE / "hf_cache")
ENV["HF_HUB_DISABLE_TELEMETRY"] = "1"
ENV["TOKENIZERS_PARALLELISM"] = "false"
if (SETUP_DIR / "tessdata").is_dir():
    ENV["PHASE3_TESSDATA"] = str(SETUP_DIR / "tessdata")
if tess := shutil.which("tesseract"):
    ENV.setdefault("PHASE3_TESSERACT", tess)
if (SETUP_DIR / "docling-venv/bin/python").is_file():
    ENV["PHASE3_DOCLING_PYTHON"] = str(SETUP_DIR / "docling-venv/bin/python")
if (SETUP_DIR / "paddle-venv/bin/python").is_file():
    ENV["PHASE3_PADDLE_PYTHON"] = str(SETUP_DIR / "paddle-venv/bin/python")
ENV["PHASE3_DOCLING_MODELS"] = str(REPO_DIR / "data/ingestion/phase3/setup/docling_models")
ENV["PHASE3_PADDLE_MODELS"] = str(REPO_DIR / "data/ingestion/phase3/setup/paddle_models")

WORKERS = 4  # Conservative initial setting for 9.6-core / 38.4 GiB quota; adjust after checking RSS.

t0 = time.perf_counter()
proc = subprocess.run([
    EMBED_PY, "-m", "ingestion", "extract",
    "--selection-manifest", str(MANIFEST),
    "--profile", str(PROFILE),
    "--workers", str(WORKERS),
    "--resume",
    "--output", str(EXTRACT_DIR),
], cwd=REPO_DIR, env=ENV)
elapsed = round(time.perf_counter() - t0, 2)
print(f"Extraction exited with code {proc.returncode} in {elapsed}s")
if proc.returncode not in (0, 1):
    raise RuntimeError(f"Extraction aborted at run level (exit {proc.returncode})")

summary = json.loads((EXTRACT_DIR / "summary.json").read_text())
print(json.dumps({
    "assets": summary["assets"],
    "excluded_assets": summary["excluded_assets"],
    "page_count": summary["page_count"],
    "page_statuses": summary["page_statuses"],
    "reused_pages": summary["reused_pages"],
    "review_items": summary["review_items"],
    "page_worker_failures": summary["page_worker_failures"],
    "peak_worker_rss_mib": summary["peak_worker_rss_mib"],
    "elapsed_seconds": summary["elapsed_seconds"],
}, indent=2))

## 6. Full Section-Aware Chunking

In [ ]:
import json, os, subprocess, sys, time
from pathlib import Path

WORKSPACE = Path(os.environ.get("MEDQUERY_WORKSPACE", "/home/jovyan/medquery_workspace"))
REPO_DIR = WORKSPACE / "repo"
SETUP_DIR = WORKSPACE / "setup"
EMBED_PY = str(SETUP_DIR / "embedding-venv/bin/python") if (SETUP_DIR / "embedding-venv/bin/python").is_file() else sys.executable
FULL_DIR = REPO_DIR / "data/ingestion/phase9/full_run"
EXTRACT_DIR = FULL_DIR / "extract"
CHUNK_DIR = FULL_DIR / "chunk"

t0 = time.perf_counter()
subprocess.run([
    EMBED_PY, "-m", "ingestion", "chunk",
    "--input", str(EXTRACT_DIR),
    "--output", str(CHUNK_DIR),
    "--strategy", "section_aware",
    "--min-tokens", "300",
    "--target-tokens", "380",
    "--max-tokens", "450",
], cwd=REPO_DIR, check=True)
print(f"Chunking finished in {round(time.perf_counter() - t0, 2)}s")

chunk_summary = json.loads((CHUNK_DIR / "chunk_summary.json").read_text())
print(json.dumps({
    "strategy": chunk_summary["strategy"],
    "params": chunk_summary["params"],
    "documents_total": chunk_summary["documents_total"],
    "documents_chunked": chunk_summary["documents_chunked"],
    "unchunkable_documents_count": chunk_summary["unchunkable_documents_count"],
    "unchunkable_documents": chunk_summary["unchunkable_documents"],
    "total_chunks": chunk_summary["total_chunks"],
    "total_spans": chunk_summary["total_spans"],
    "total_tokens": chunk_summary["total_tokens"],
}, indent=2))

## 7. Full GPU Embeddings (E5-small on CUDA)

In [ ]:
import json, os, subprocess, sys, time
from pathlib import Path

WORKSPACE = Path(os.environ.get("MEDQUERY_WORKSPACE", "/home/jovyan/medquery_workspace"))
REPO_DIR = WORKSPACE / "repo"
SETUP_DIR = WORKSPACE / "setup"
EMBED_PY = str(SETUP_DIR / "embedding-venv/bin/python") if (SETUP_DIR / "embedding-venv/bin/python").is_file() else sys.executable
FULL_DIR = REPO_DIR / "data/ingestion/phase9/full_run"
CHUNK_DIR = FULL_DIR / "chunk"
EMBED_DIR = FULL_DIR / "embed"

ENV = os.environ.copy()
ENV["HF_HOME"] = str(WORKSPACE / "hf_cache")
ENV["HF_HUB_DISABLE_TELEMETRY"] = "1"
ENV["TOKENIZERS_PARALLELISM"] = "false"

GPU_BATCH_SIZE = 128  # Automatic OOM halving + CPU fallback remain active if needed.

t0 = time.perf_counter()
subprocess.run([
    EMBED_PY, "-m", "ingestion", "embed",
    "--input", str(CHUNK_DIR),
    "--output", str(EMBED_DIR),
    "--model", "intfloat/multilingual-e5-small",
    "--revision", "614241f622f53c4eeff9890bdc4f31cfecc418b3",
    "--device", "cuda",
    "--gpu-batch-size", str(GPU_BATCH_SIZE),
    "--local-files-only",
], cwd=REPO_DIR, env=ENV, check=True)
print(f"Embedding finished in {round(time.perf_counter() - t0, 2)}s")

emb_summary = json.loads((EMBED_DIR / "embedding_summary.json").read_text())
print(json.dumps(emb_summary, indent=2))

## 8. Final Artifact Verification

In [ ]:
import json, os, sys
from pathlib import Path

WORKSPACE = Path(os.environ.get("MEDQUERY_WORKSPACE", "/home/jovyan/medquery_workspace"))
REPO_DIR = WORKSPACE / "repo"
INPUT_DIR = WORKSPACE / "input"
FULL_DIR = REPO_DIR / "data/ingestion/phase9/full_run"
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

from ingestion.transfer import verify_phase9_outputs

report = verify_phase9_outputs(
    INPUT_DIR / "manifests/hpc_runtime_manifest.json",
    FULL_DIR / "extract",
    FULL_DIR / "chunk",
    FULL_DIR / "embed",
)
extract_sum = json.loads((FULL_DIR / "extract/summary.json").read_text())
embed_sum = json.loads((FULL_DIR / "embed/embedding_summary.json").read_text())
report["timings_and_resources"] = {
    "extract_elapsed_seconds": extract_sum.get("elapsed_seconds"),
    "extract_peak_worker_rss_mib": extract_sum.get("peak_worker_rss_mib"),
    "embed_elapsed_seconds": embed_sum.get("elapsed_seconds"),
    "embed_inference_seconds": embed_sum.get("inference_seconds"),
    "embed_peak_gpu_memory_mib": embed_sum.get("peak_gpu_memory_mib"),
    "embed_peak_rss_mib": embed_sum.get("peak_rss_mib"),
}
(FULL_DIR / "verification_report.json").write_text(json.dumps(report, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
print(json.dumps(report, ensure_ascii=False, indent=2))
assert report["storage_compatible"], "Storage compatibility check failed!"

## 9. Package Downloadable Results Archive

In [ ]:
import hashlib, os, zipfile
from pathlib import Path

WORKSPACE = Path(os.environ.get("MEDQUERY_WORKSPACE", "/home/jovyan/medquery_workspace"))
REPO_DIR = WORKSPACE / "repo"
INPUT_DIR = WORKSPACE / "input"
FULL_DIR = REPO_DIR / "data/ingestion/phase9/full_run"
ARCHIVE_PATH = WORKSPACE / "medquery_hpc_results.zip"

FILES_TO_PACKAGE = [
    ("manifests/extraction_selection_manifest.json", INPUT_DIR / "manifests/extraction_selection_manifest.json"),
    ("manifests/transfer_inventory.json", INPUT_DIR / "manifests/transfer_inventory.json"),
    ("manifests/portable_path_map.json", INPUT_DIR / "manifests/portable_path_map.json"),
    ("extract/pages.jsonl", FULL_DIR / "extract/pages.jsonl"),
    ("extract/sections.jsonl", FULL_DIR / "extract/sections.jsonl"),
    ("extract/review_queue.jsonl", FULL_DIR / "extract/review_queue.jsonl"),
    ("extract/documents.json", FULL_DIR / "extract/documents.json"),
    ("extract/attempts.json", FULL_DIR / "extract/attempts.json"),
    ("extract/summary.json", FULL_DIR / "extract/summary.json"),
    ("extract/identity.json", FULL_DIR / "extract/identity.json"),
    ("chunk/chunks.jsonl", FULL_DIR / "chunk/chunks.jsonl"),
    ("chunk/chunk_spans.jsonl", FULL_DIR / "chunk/chunk_spans.jsonl"),
    ("chunk/chunk_summary.json", FULL_DIR / "chunk/chunk_summary.json"),
    ("embed/embedding_model.json", FULL_DIR / "embed/embedding_model.json"),
    ("embed/chunk_embeddings.jsonl", FULL_DIR / "embed/chunk_embeddings.jsonl"),
    ("embed/embedding_summary.json", FULL_DIR / "embed/embedding_summary.json"),
    ("verification_report.json", FULL_DIR / "verification_report.json"),
]

def sha256_file(path):
    h = hashlib.sha256()
    with path.open("rb") as f:
        while b := f.read(1024 * 1024):
            h.update(b)
    return h.hexdigest()

checksums = []
with zipfile.ZipFile(ARCHIVE_PATH, "w", compression=zipfile.ZIP_DEFLATED, compresslevel=6) as zf:
    for arcname, src in FILES_TO_PACKAGE:
        if not src.is_file():
            raise FileNotFoundError(f"Missing required result file: {src}")
        zf.write(src, arcname=arcname)
        checksums.append(f"{sha256_file(src)}  {arcname}")
    zf.writestr("results_checksums.sha256", "\n".join(checksums) + "\n")

with zipfile.ZipFile(ARCHIVE_PATH, "r") as zf:
    bad = zf.testzip()
    if bad is not None:
        raise RuntimeError(f"Corrupt member in results zip: {bad}")

archive_sha = sha256_file(ARCHIVE_PATH)
(WORKSPACE / "medquery_hpc_results.zip.sha256").write_text(f"{archive_sha}  {ARCHIVE_PATH.name}\n", encoding="utf-8")
print(f"Created {ARCHIVE_PATH} ({ARCHIVE_PATH.stat().st_size / 1024**2:.2f} MiB)")
print(f"SHA-256: {archive_sha}")